# REDUCE and EXTRACT fixed slits for all LEGGOS NIRSpec datasets (prism and gratings)
I've organized all the raw files into directories by target, and then by date of observation.  2022-08-13/, 2023-01-13/, and so forth, each with the *uncal.fits files.  Modifed a version of the notebook I ran for CDFS and COSMOS, so it will all run in batch mode.
jrigby, Jan 2026.  Original version from B. Welch  
Feb 2026:  Modified to run on multiple processors, which is a lot faster (but sometimes crashes the computer).
Followed [example 2](here https://jwst-pipeline.readthedocs.io/en/1.20.0/jwst/user_documentation/running_pipeline_python.html#multiprocessing)



Regression testing, upgrading from pipeline v1.20.2 to v2.0.1.  The main changes are that NSClean gets 
applied earlier, and there's now picture frame correction.

Trying prism first.   Let's power through this

In [146]:
import os
# 1) where the jwst pipeline config files are located
home = "/Users/jrrigby1/Ref_files/"

# STScI helpdesk says these os commands need to come BEFORE jwst pipeline packages are imported
os.environ["CRDS_PATH"] = home + "crds_cache/jwst_ops"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
os.environ["CRDS_CONTEXT"] =   'jwst_1464.pmap'     # for  pipeline v1.20.2, Fall 2025
os.environ["CRDS_CONTEXT"] =   'jwst_1535.pmap'      # For pipeline v2.0.1 , Spring 2026

In [147]:
print("Computer has this many CPUs", int(os.cpu_count()))
cores2use = 4  # Ran out of memory 3/5/2026 with N=5.  If rate files are really big (>~1GB, then easy to run out of memory)
print("Will use", cores2use, "cores")

Computer has this many CPUs 14
Will use 4 cores


In [148]:
# Avoid re-running parts that finished  *** MODIFY AS NEEDED
pipev = '2.0.1'  # Which version of the pipeline are we running
pipev = '1.20.2'
RUN_LEVEL0  = True
RUN_LEVEL1  = True
RUN_LEVEL2  = True
RUN_LEVEL3  = True
RUN_EXTRACT = True

In [149]:
import numpy as np
import glob
from multiprocessing.pool import Pool
from os.path import basename, dirname, normpath
import matplotlib.pyplot as plt
import pandas
import re
from jrr.jrjwst import writel3asn, extract1D_SB  # Tools from Brian Welch, David Law to make associations
from jrr.jrjwst import   wrap_median_combine_level3_nirspecFS # special median combine and extraction for FS backgrounds
from jrr.spec import mark_CaII_Fraunhofer_lines
from jrr.util import gethead
from jrmulti import run_jwst_det1   # If running mulitprocess in a Jupyter notebook, the function fed to starmap must be a .py file
from jrmulti import run_jwst_spec2  # If running mulitprocess in a Jupyter notebook, the function fed to starmap must be a .py file
from jrmulti import  run_jwst_custom_extraction
from collections import defaultdict
plt.rcParams["figure.figsize"] = (9,4)
import warnings
warnings.filterwarnings('ignore') 

In [150]:
import json
from astropy.io import fits
from astropy.utils.data import download_file
import astropy.units as u
from astropy import wcs
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import matplotlib as mpl

# The calwebb_spec and spec3 pipelines
#from jwst.pipeline import Spec2Pipeline  # This is imported within jrmulti.run_jwst_spec2, docs say to prevent memory leak
from jwst.pipeline import Spec3Pipeline

import jwst
# the level1 pipeline:
#from jwst.pipeline import Detector1Pipeline   # This is imported within jrmulti.run_jwst_det1, docs say to prevent memory leak

# data models
from jwst import datamodels

# association file utilities
from jwst.associations import asn_from_list as afl # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base # Definition of a Lvl3 association file

In [151]:
# This had better be 1.20.2, or 2.0X, or STOP
if jwst.__version__ != pipev:
    raise Exception("ERROR, pipeline version", jwst.__version__, 'is not the expectation, pipev')
else : print('Good, pipeline version was as expected:', jwst.__version__)    

Good, pipeline version was as expected: 1.20.2


In [152]:
def poll_spectrum_at_wavelength(wave, spectrum, atwave=3.0):  # find the value of the spectrum at wavelength atwave
    idx = (np.abs(wave - atwave)).argmin()
    return spectrum[idx]

In [153]:
#leggos_in  = '/Volumes/Rawdata/NIRSpec_Fixedslit/CDFS_raw_G140M/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/' 
#leggos_in =  '/Volumes/Rawdata/Raw_NIRSpecFS_for_PSF_measurement/G395M/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpecFS_reduced_PSF_subsamp_x4/G395M/'
#leggos_in = '/Volumes/Rawdata/NIRSpec_Fixedslit/CDFS_raw_prism/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSPEC_regression2/'
#leggos_in = '/Volumes/JWST_bkgs_prism/Raw_prism_May2026/'
#leggos_out = '/Volumes/JWST_bkgs_prism/Reduced_prism_May2026/'
leggos_in = '/Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/'
leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/'
epochdirs = glob.glob(leggos_in + '/*/*/')

In [154]:
#rerun_these = ['2024-11-22',] # '2023-12-08', '2024-01-06', '2023-05-22', '2023-12-02', '2024-11-21', '2023-12-04', '2023-12-09']
#dirs_to_handle = [a for a in epochdirs if any(b in a for b in rerun_these)]

In [155]:
# Select which epochs to reduce.  All, or subset.  ***** MODIFY THIS *****
#dirs_to_handle = [x for x in epochdirs if 'VID01254006001' in x]
#dirs_to_handle = [leggos_in + x for x in missingL3]
#dirs_to_handle = epochdirs
dirs_to_handle = [x for x in epochdirs if '2022-10-22' in x]
dirs_to_handle

['/Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/']

In [156]:
# Organize the output folders
folder_L2a = 'L2a/'
folder_L2b = 'L2b_pipeclean/'
folder_L3  = 'L3_destripe/'
folder_extracted = 'L3_extracted/'
output_folders = [folder_L2a, folder_L2b, folder_L3, folder_extracted]

In [157]:
# Make some dictionaries to hold files
raw_files = {}
l2a_files = {}
allmsa_calfiles = {}
allmsa_asnfile = {}

In [158]:
def sort_out_paths(thisdir, reduced_datadir, pipev='1.20.2'):  # Used several times, so generalize this 
    targname = thisdir.split('/')[-2]   # this is targname for pipev=1.20.2, and visitID for pipev=2
    input_path = thisdir  
    if pipev=='1.20.2':
        epochname =   basename(re.sub(r'\/$', '', thisdir))
        output_path = reduced_datadir + targname + '/' + epochname + '/'     # where to write result 
    elif pipev=='2.0.1':
        epochname = thisdir.split('/')[-3]   
        targname =  thisdir.split('/')[-2]   # this is visitID for pipev=2
        output_path = reduced_datadir + epochname + '/' + targname + '/' 
    return(epochname, targname, input_path, output_path)

In [159]:
if RUN_LEVEL0 :   # prep the directories we'll need
    for ii, thisdir in enumerate(dirs_to_handle) :   
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        if os.path.exists(output_path) == False: # if folder doesn't exist
            print('   Creating folder ' + output_path)
            if pipev =='1.20.2':
                os.system('mkdir '  + leggos_out + '/' + targname) 
                os.system('mkdir '  + output_path) # creates the folder
            elif pipev == '2.0.1':                
                os.system('mkdir '  + leggos_out + '/' + epochname) 
                os.system('mkdir '  + output_path) # creates the folder
        
        for folder in output_folders :       #Make output folders if they don't already exist
            if os.path.exists(output_path + folder) == False: # if folder doesn't exist
                #print('   Creating folder ' + output_path + folder)
                os.system('mkdir ' + output_path + folder) # creates the folder

   Creating folder /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/


In [160]:
if RUN_LEVEL0 :   # FOOL THE PIPELINE:  Make the data look like they were taken in fixed slit mode, when they were not.
    for ii, thisdir in enumerate(dirs_to_handle) :   
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)           
        raw_files[ii] = glob.glob(input_path + '*nrs*_uncal.fits') #list the uncalibrated (level 1b) files.
        raw_files[ii] = sorted(raw_files[ii])
        counter = 0
        for exposure in raw_files[ii]:
            test = fits.open(exposure)
            head = test[0].header
            if head["EXP_TYPE"] == 'NRS_LAMP' : raise Exception('ERROR! There should not be any EXP_TYPE NRS_LAMP frames!')
            elif head["EXP_TYPE"] != "NRS_FIXEDSLIT":
                test[0].header["EXP_TYP0"] = head["EXP_TYPE"]  # Book-keep the original exp_type
                test[0].header["EXP_TYPE"] = "NRS_FIXEDSLIT"   # Modify the EXP_TYPE.  Kludge!
                counter += 1
            if pipev=='2.0.1':  # Extra workaround for bug in pipeline v2.0.1, per helpdesk ticket 
                if head['OPMODE'] != 'FIXEDSLIT' :
                    test[0].header['OPMODE0'] = head['OPMODE']  # Book-keep 
                    test[0].header['OPMODE'] = 'FIXEDSLIT'      # short-term kludge, per helpdesk ticket INC0222900
                    counter += 1
            test.writeto(exposure, overwrite=True)
        print(thisdir, "dir", ii, "of", len(dirs_to_handle) - 1, ", Checked", len(raw_files[ii]), "files, and changed ", counter, 'keywords')

/Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/ dir 0 of 0 , Checked 6 files, and changed  0 keywords


In [161]:
if RUN_LEVEL1 :
    print('Starting Stage 1 reductions!')
    if pipev == '1.20.2':
        param_nested_dict = {'jump':{'expand_large_events': True}}
    elif pipev == '2.0.1':
        param_nested_dict = {'jump':{'expand_large_events': True}, 'picture_frame': {'skip': False}, \
                    'clean_flicker_noise':{'skip': False, 'autoparam': False, 'fit_method': 'fft', \
                    'background_method': None, 'n_sigma': 2, 'mask_science_regions': True, 'save_noise': False}}
        # fit_method='fft' is how you now invoke nsclean under pipeline v2
        # I have tried to set up the parameters to what nsclean expects, not sure I've done it right.  -JR
        # save_noise=True is useful for debugging, to check that 1/f noise has been removed.  But will fill your hard drive
    else : raise Exception("ERROR, do not have parameters for this pipeline version", pipev)

    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        print("Reducing visitID, epoch", targname, epochname, "which is dir", ii+1, "of", len(dirs_to_handle))
        rawfiles_thisepoch = glob.glob(input_path + '*nrs*_uncal.fits') #list the uncalibrated (level 1b) files.
        output_dir = output_path + folder_L2a
        outptd = [output_dir for _ in range(len(rawfiles_thisepoch))]
        list_of_dicts = [param_nested_dict for _ in range(len(rawfiles_thisepoch))]

        with Pool(cores2use) as pool:
            pool.starmap(run_jwst_det1, zip(rawfiles_thisepoch, outptd, list_of_dicts))

Starting Stage 1 reductions!
Reducing visitID, epoch 2022-10-22 2022-10-22 which is dir 1 of 1


/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/asdf/_asdf.py:286: AsdfPackageVersionWarning: File 'file:///Users/jrrigby1/Ref_files/crds_cache/jwst_ops/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf'was created with extension URI 'asdf://asdf-format.org/core/extensions/core-1.6.0' (from package asdf==5.1.0), but older package (asdf==5.0.0) is installed.
  warnings.warn(msg, AsdfPackageVersionWarning)
/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/asdf/_asdf.py:312: AsdfPackageVersionWarning: File 'file:///Users/jrrigby1/Ref_files/crds_cache/jwst_ops/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf'was created with package asdf_standard==1.5.0, but older package(asdf_standard==1.4.0) is installed.
  warnings.warn(msg, AsdfPackageVersionWarning)
/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/asd

Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00003_nrs2_uncal.fits
Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00002_nrs2_uncal.fits
Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00002_nrs1_uncal.fits
Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00004_nrs2_uncal.fits
Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00004_nrs1_uncal.fits
Pipeline ran:  /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/jw02767005001_03101_00003_nrs1_uncal.fits


In [162]:
if RUN_LEVEL2 :
    print('Starting Stage 2 reductions!')
    if pipev == '1.20.2':
        param_nested_dict = {"nsclean":{'skip': False}, 'srctype':{'source_type': 'EXTENDED'}}  # for pipeline v1.20.2
    elif pipev == '2.0.1':
        param_nested_dict = {'srctype':{'source_type': 'EXTENDED'}}   # for pipeline v2.0.1
    else: raise Exception("ERROR, do not have parameters for this pipeline version", pipev)
    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        print("DEBUG", epochname, targname, input_path, output_path)
        files_to_run = glob.glob(output_path + folder_L2a + '*nrs*_rate.fits')
        print('Reducing dir', epochname, targname, 'which is dir', ii+1, 'of', len(dirs_to_handle))
        output_dir =  output_path + folder_L2b
        outptd = [output_dir for _ in range(len(files_to_run))]
        list_of_dicts = [param_nested_dict for _ in range(len(files_to_run))]

        with Pool(cores2use) as pool:
            pool.starmap(run_jwst_spec2, zip(files_to_run, outptd, list_of_dicts))

Starting Stage 2 reductions!
DEBUG 2022-10-22 2022-10-22 /Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/ /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/
Reducing dir 2022-10-22 2022-10-22 which is dir 1 of 1


/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/jwst/clean_flicker_noise/lib.py:202: RuntimeWarning: divide by zero encountered in matmul
  pinv_pb = np.matmul(np.linalg.inv(np.matmul(_a_h, _a)), _a_h)
/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/jwst/clean_flicker_noise/lib.py:202: RuntimeWarning: overflow encountered in matmul
  pinv_pb = np.matmul(np.linalg.inv(np.matmul(_a_h, _a)), _a_h)
/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/jwst/clean_flicker_noise/lib.py:202: RuntimeWarning: invalid value encountered in matmul
  pinv_pb = np.matmul(np.linalg.inv(np.matmul(_a_h, _a)), _a_h)
/Users/jrrigby1/miniforge3/envs/JWSTDP-1.20.2-1-py312-macos-arm64/lib/python3.12/site-packages/jwst/clean_flicker_noise/lib.py:202: RuntimeWarning: divide by zero encountered in matmul
  pinv_pb = np.matmul(np.linalg.inv(np.matmul(_a_h, _a)), _a_h)
/Users/jrri

Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00003_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00002_nrs1_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00004_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00003_nrs1_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00002_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L2a/jw02767005001_03101_00004_nrs1_rate.fits


In [163]:
if RUN_LEVEL3:  #   Level 3 processing for each epoch
    for ii, thisdir in enumerate(dirs_to_handle):  
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        L3_destripedir = output_path + folder_L3
        L3_extractdir  = output_path + folder_extracted
        print("working on", epochname, targname, "which is dir", ii+1, "of", len(dirs_to_handle))
        for foo in (L3_destripedir, L3_extractdir):
            if os.path.exists(foo) == False:       # make a L3_destripe and L3_extract directory
                print('Creating folder ' + foo)
                os.system('mkdir ' + foo) # creates the folder
            # Write the associations needed to extract the spectra
        allmsa_calfiles  = glob.glob(output_path + folder_L2b + '/*cal.fits') 
        print("allmsa_calfiles has len", len(allmsa_calfiles))
        allmsa_asnfile   = L3_destripedir + 'L3asn.json'
        writel3asn(allmsa_calfiles, None, allmsa_asnfile, epochname + '_' + targname) 
        spec3 = Spec3Pipeline()
        spec3.output_dir = L3_destripedir
        spec3.outlier_detection.skip = False
        #spec3.resample_spec.pixel_scale_ratio = 4.0  # Subsample, when I needed to measure width of trace to get PSF
        #spec3.extraction_type = 'optimal'     #  EXPERIMENT for LSF PN calibration program. 
        spec3.save_results = True 
        spec3.run(allmsa_asnfile)

working on 2022-10-22 2022-10-22 which is dir 1 of 1
allmsa_calfiles has len 6


In [164]:
# Not sure why this isn't making spectra!
#if RUN_EXTRACT:   # Same as below, but running on more cores
#    cores_for_extraction = 8
#    for ii, thisdir in enumerate(dirs_to_handle): 
#        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
#        indir = output_path + folder_L3
#        outdir  = output_path + folder_extracted
#        print(thisdir, "dir", ii, "of", len(dirs_to_handle) -1)
#        with Pool(cores_for_extraction) as pool:
#            pool.starmap(run_jwst_custom_extraction, zip(indir, outdir))

In [165]:
if RUN_EXTRACT:
    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        L3_destripedir = output_path + folder_L3
        L3_extractdir  = output_path + folder_extracted
        print("Extracting spectra for dir ", ii, 'of', len(dirs_to_handle) -1, targname, epochname)
        # This runs Jane's custom median combine for backgrounds
        try:
            spectra = wrap_median_combine_level3_nirspecFS(L3_destripedir, L3_extractdir)
        except: print(thisdir, "FAILED at wrap_median_combine")

Extracting spectra for dir  0 of 0 2022-10-22 2022-10-22
/Volumes/Rawdata/NIRSpec_Fixedslit/Rosalia_G140M/NoNameF070LP_G140M/2022-10-22/ FAILED at wrap_median_combine


In [166]:
L3_extractdir

'/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/2022-10-22/2022-10-22/L3_extracted/'

In [167]:
targname

'2022-10-22'

In [168]:
epochname

'2022-10-22'

## Now, go see MSA_FixedSlit_analysis_plotting.ipynb, to plot results

In [169]:
if RUN_EXTRACT:
    fraun = False
    if fraun:  mark_CaII_Fraunhofer_lines()
    #for key in [x for x in spectra.keys() if 's200a' in x] :
    for key in [x for x in spectra.keys() if (('1600' in x))]:
        plt.step(wave_array[key], spectra[key], label=key, lw=1, where='mid')
        ## Two spectra have high backgrounds, what happened?  Pull them
        #val_3um = poll_spectrum_at_wavelength(wave_array[key], spectra[key])
        #if val_3um > 0.2:
        #    print(key, val_3um)
    plt.xlabel("wavelength (micron)")
    plt.ylabel("surface brightness (MJy/sr)")
    #plt.title("Background, NIRSpec fixed slit")
    if fraun:  
        plt.xlim(0.8, 0.92)
        plt.ylim(0.0,1)
    plt.legend(fontsize=8)
    #plt.ylim(0,30)
    #plt.xlim(0.7, 0.85)

NameError: name 'spectra' is not defined

In [ ]:
spectra.keys()